Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: GCP 상에서 최근 발생한 조직 정책(Organization Policy ) 제약 조건 위반 실패 감사 로그를 실시간으로 역추적하여, 제약 사항을 즉시 해결하기 위해 해결 명령어를 처방하는 도구다.

## 조직 정책 위반 해결사 (Organization Policy Resolver )

### 1. 의존성 패키지 설치

GCP 로깅 감사 로그를 정밀 분석하기 위해 필요한 핵심 클라우드 라이브러리를 설치한다.

In [ ]:
!pip install --quiet google-cloud-logging

### 2. 활성 GCP 프로젝트 ID 동적 탐색

현재 활성화되어 있는 사용자의 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색한다. 자격 증명이 유효하지 않은 경우 수동으로 구성할 수 있다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )


In [ ]:
import google.auth

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id" # 본인의 실제 GCP 프로젝트 ID로 변경하기 바란다.


### 3. 조직 정책 위반 감사 로그 수집 및 정밀 진단

GCP 활동 감사 로그로부터 조직 정책 제약 조건 위반 실패 내역을 역추적하여, 해당 실패 사유에 알맞은 임시 우회 방안과 즉시 실행 가능한 gcloud 처방 명령어를 출력한다.

**상세 분석 흐름 및 규칙:**
* **감사 로그 필터링**: 감사 로그 검색 시 에러 코드가 9(Failed Precondition )이거나 상태 메시지에 제약 조건 위반(`Constraint`, `violated` ) 문구가 포함된 활동 감사 로그만 수집한다.
* **제약 조건 ID 동적 추출**: 실패 설명 본문에서 정규 표현식을 사용하여 실제 제한이 가해진 제약 조건 이름(예: `constraints/iam.disableServiceAccountKeyCreation`, `constraints/compute.vmExternalIpAccess`, `constraints/storage.publicAccessPrevention` )을 동적으로 파싱한다.
* **처방 및 임시 조치 명령 조립**: 위반된 제약 사항의 위험성을 알리는 경고 정보와 함께, 우회가 꼭 필요한 개발자들을 위해 해당 조직 정책 제약을 일시적으로 완화하는 `resource-manager org-policies disable-enforce` 명령을 출력한다.

In [ ]:
from datetime import datetime, timedelta, timezone
from google.cloud import logging_v2
import re

DAYS = 7
LIMIT_COUNT = 5

client = logging_v2.Client(project=project_id)
start_date = (datetime.now(timezone.utc) - timedelta(days=DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")

log_filter = (
  f'logName="projects/{project_id}/logs/cloudaudit.googleapis.com%2Factivity" '
  f'AND (protoPayload.status.code=9 OR protoPayload.status.message:"Constraint" OR protoPayload.status.message:"violated") '
  f'AND timestamp>="{start_date}"'
)

print(f"[자가 진단 시작] 최근 발생한 GCP 조직 정책 위반 실패 감사 로그 역추적을 시작한다...")
print(f"조회 대상 기간: 최근 {DAYS}일")
print(f"조회 대상 로그 개수: 최대 {LIMIT_COUNT}개\n" + "-"*72)

try:
  entries = client.list_entries(filter_=log_filter, order_by="timestamp desc", page_size=LIMIT_COUNT)
  found_attempts = 0
  
  for entry in entries:
    if found_attempts >= LIMIT_COUNT:
      break
      
    payload = entry.proto_payload
    if not payload:
      continue
      
    principal_email = payload.get("authenticationInfo", {}).get("principalEmail")
    service_name = payload.get("serviceName")
    method_name = payload.get("methodName")
    status_message = payload.get("status", {}).get("message", "상세 사유 누락")
    
    if not principal_email or not service_name:
      continue
      
    found_attempts += 1
    print(f"[위반 실패 건 #{found_attempts}]")
    print(f"  - 실패 주체 계정: {principal_email}")
    print(f"  - 요청 대상 서비스: {service_name}")
    print(f"  - 실행 실패 액션: {method_name}")
    print(f"  - 실제 오류 내용: {status_message}\n")
    
    constraint_match = re.search(r"constraints/[a-zA-Z0-9.]+", status_message)
    constraint_id = constraint_match.group(0) if constraint_match else "constraints/unknown"
    short_constraint = constraint_id.replace("constraints/", "")
    
    policy_desc = "조직 정책에 의해 설정된 자원 생성 및 제어 규약 제약 사항이다."
    recommended_action = "보안 정책상 위반 상태다. 우회가 필요한 경우 아래 제약 해제 명령을 검토하기 바란다."
    
    if constraint_id == "constraints/iam.disableServiceAccountKeyCreation":
      policy_desc = "서비스 계정 키 생성을 차단하는 제약 사항이다."
      recommended_action = "안전한 인증을 위해 워크로드 아이덴티티(Workload Identity ) 연동 권장하나, 테스트 목적상 키 발급이 필요하면 제약을 해제할 수 있다."
    elif constraint_id == "constraints/compute.vmExternalIpAccess":
      policy_desc = "VM 인스턴스에 외부 IP 주소 할당을 차단하는 제약 사항이다."
      recommended_action = "내부 IP 전용 VM 사용 및 Cloud NAT 구성을 권장하나, 외부 IP가 필요한 경우 제약을 해제할 수 있다."
    elif constraint_id == "constraints/storage.publicAccessPrevention":
      policy_desc = "스토리지 버킷의 공용 공개 액세스를 차단하는 제약 사항이다."
      recommended_action = "비공개 액세스 유지를 권장하나, 정적 웹사이트 호스팅 등 공용 배포가 필요할 경우 제약을 해제할 수 있다."
      
    print(f"  [정밀 처방 제약 조건 분석]")
    print(f"    - 검출 제약 사항: {constraint_id}")
    print(f"    - 제약 조건 설명: {policy_desc}")
    print(f"    - 권장 임시 방안: {recommended_action}\n")
    
    if short_constraint != "unknown":
      print(f"  [즉시 조치 가능한 원클릭 gcloud 해결 명령어]")
      print(f"    gcloud resource-manager org-policies disable-enforce \"{short_constraint}\" \\")
      print(f"      --project=\"{project_id}\"")
    else:
      print(f"  [즉시 조치 가능한 가이드]")
      print(f"    실제 거부 사유 구문에서 구체적인 제약 명칭을 파싱하지 못했다. 상단의 에러 로그를 점검하여 조직 정책에서 알맞은 예외 정책을 매핑하기 바란다.")
    print("-" * 72)
    
  if found_attempts == 0:
    print(f"[성공] GCP 조직 정책 제약 조건 위반 실패 감사 로그 사례가 발견되지 않았다. 안전하다!")
    print(f"     (GCP 조직 정책 설정 콘솔 주소: https://console.cloud.google.com/iam-admin/orgpolicies )")
    
except Exception as e:
  print(f"[오류] 로그를 조회하는 중 에러가 발생했다. 자격 증명 또는 권한 설정을 점검하기 바란다: {e}")
